In [1]:
import openai
from openai import AsyncOpenAI
from agents.models import openai_provider
from agents import (
set_default_openai_api,
set_default_openai_client,
set_tracing_disabled
)
import asyncio
import os
from agents import Agent,Runner,function_tool

# 对模型进行配置文件的设置，之后可以直接使用

api_key = os.environ["DEEPSEEK_API_KEY"]
url = os.environ["DEEPSEEK_API_BASE_URL"]

client = AsyncOpenAI(api_key= api_key,base_url = url)

# 使用自定义客户端
set_default_openai_client(client = client, use_for_tracing = False)

# 使用兼容的API模式
set_default_openai_api("chat_completions")

# 禁用OpenAI跟踪服务
set_tracing_disabled(disabled = True)

model_name = 'deepseek-chat'

openai_provider.DEFAULT_MODEL = model_name # 自定义模型名称


In [16]:
# 构建多个AGENT，使用路由编排
agent_A = Agent(
    name = "deepseek_chatA",
    model = "deepseek-chat",
    instructions = "你是一个AI助手A，负责用俄语进行回答"
)
agent_B = Agent(
    name = "deepseek_chatB",
    model = "deepseek-chat",
    instructions = "你是一个AI助手B，负责用法语进行回答"
)
agent_C = Agent(
    name = "deepseek_chatC",
    model = "deepseek-chat",
    instructions = "你是一个AI助手C，负责用德语进行回答"
)
agent_D = Agent(
    name = "前台助手",
    model = "deepseek-chat",
    instructions = "你是一个前台助手D，不回答问题，只需要根据用户使用的语言类型将问题交付给适合回答的Agent即可",
    handoffs=[agent_A,agent_B,agent_C], # 让模型知道有那些可以被交付的Agent
)

In [15]:
async def main():
    """
    路由编排A、B、C，只有一个agent接收用户问题，负责将用户问题交付给适合的agent进行回答
    """
    result = await Runner.run(agent_D, input="Translate I like you to Chinese")
    print("Agent_D:\n",result.final_output)

await main()

Agent_D:
 我喜欢你
